# Single-application placement in Colab
Use an A100-class GPU with enough free memory for the pinned checkpoint. This notebook preserves runtime, package, source, test, and scenario verification gates. Genuine inference is not verified by a reference result.

## 1. Stage your configured repository
Upload the complete cleaned repository to the Drive folder below, including tests and their fixtures. Edit `config/application.yaml` and supporting inputs before staging. Staging creates a new local directory; it never overwrites the Drive project or previous runs.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys, uuid
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/optimind-gurobi-workload-placement')
RUNTIME_PROJECT = Path('/content') / ('placement-' + uuid.uuid4().hex[:12])
shutil.copytree(DRIVE_PROJECT, RUNTIME_PROJECT, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__', '.pytest_cache', '.model_cache', 'results'))
os.chdir(RUNTIME_PROJECT)
SCENARIO = RUNTIME_PROJECT / 'config/scenario.yaml'
RESULTS = RUNTIME_PROJECT / 'results'
CACHE = Path('/content/optimind-model-cache')


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=True,
)

## 2. Choose mode and install dependencies
Start with reference mode. For AI, set USE_AI=True and rerun Steps 2–4; do not restage merely to switch modes. Install the pinned text-only stack, without torchvision.


In [ ]:
USE_AI = False
RUN_PIPELINE = False
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
if USE_AI:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-inference-transformers.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'src.text_environment', '--output', str(RESULTS/'text_environment.json')], check=True)


## 3. Verify this runtime and application
Runs the complete suite and independent trusted-formulation verification for the selected scenario. No model inference. Inputs, source, hardware, suite membership and hashes are bound to the runtime evidence.


In [ ]:
from src.runtime_verification import verify_runtime, sync_verified
RUNTIME_EVIDENCE, report = verify_runtime(RESULTS/'runtime', DRIVE_PROJECT/'results'/RUNTIME_PROJECT.name/'runtime', SCENARIO)
assert report['ready_for_inference'] and report['colab_verification'] == 'VERIFIED', report['readiness_reasons']
print('Runtime evidence:', RUNTIME_EVIDENCE)
print('Tests:', report['tests']['passed'], 'Scenario verification:', report['benchmarks']['application']['passed'])


## 4. Review and authorize one launch
After inspecting verification results, change RUN_PIPELINE to True in this cell. Model access and hardware/token-budget preflight must still pass. Keep historical runs intact.


In [ ]:
RUN_PIPELINE = False  # Change to True only after reviewing Step 3.


## 5. Launch once
This consumes the launch flag and prevents overlapping notebook launches. AI errors remain errors; there is no reference fallback after AI failure.


In [ ]:
if not RUN_PIPELINE:
    print('Prepared only. Review verification and authorize in Step 4.')
else:
    RUN_PIPELINE = False
    for process in Path('/proc').iterdir():
        if process.name.isdigit():
            try:
                args = (process/'cmdline').read_bytes().split(b'\0')
                assert not any(m in args for m in (b'src.inference_worker', b'src.pipeline')), 'Existing inference worker/controller'
            except (FileNotFoundError, ProcessLookupError):
                pass
    lock = RUNTIME_PROJECT / '.placement-launch.lock'
    fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.close(fd)
    launch = RESULTS / ('launch-' + uuid.uuid4().hex[:12])
    try:
        command = [sys.executable, '-m', 'src.pipeline', '--config', str(SCENARIO),
                   '--mode', 'AI_ASSISTED' if USE_AI else 'REFERENCE_FALLBACK',
                   '--ai-config', str(RUNTIME_PROJECT/'config/optimind.yaml'),
                   '--runtime-evidence', str(RUNTIME_EVIDENCE), '--cache-dir', str(CACHE), '--output', str(launch)]
        completed = subprocess.run(command)
        print('Exit:', completed.returncode)
    finally:
        lock.unlink()
        sync_verified(RESULTS, DRIVE_PROJECT/'results'/RUNTIME_PROJECT.name/'runs')


## 6. Inspect the new result
Require RECOMMENDED, a valid DEPLOYABLE independent audit and proven optimality. For AI additionally require OPTIMIND_SPEC_TRUSTED_MILP, genuine_inference=true and fallback_used=false. No inference runs in this cell.


In [ ]:
assert 'launch' in globals(), 'Launch the pipeline first.'
for path in launch.glob('*/recommendation.json'):
    result = json.loads(path.read_text())
    print(path)
    print(json.dumps({k: result[k] for k in ('status', 'source', 'genuine_inference', 'fallback_used', 'placement', 'metrics', 'errors')}, indent=2))
